# NMF on ModulAir 00684

In [24]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [25]:
#importing data from Modulair MOD-00684
data = pd.read_csv('/content/MOD-00684.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:42Z,577611581,2025-12-31T18:59:42Z,MOD-00684,49.4,0.1,11.578,0.908,0.229,0.048,0.052,...,33.422,25.941,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,4.20
2025-12-31T23:58:42Z,577611580,2025-12-31T18:58:42Z,MOD-00684,49.5,0.1,11.323,0.940,0.248,0.054,0.089,...,33.418,25.930,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,4.09
2025-12-31T23:57:42Z,577611578,2025-12-31T18:57:42Z,MOD-00684,49.8,0.1,12.160,1.016,0.181,0.042,0.057,...,34.116,25.193,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,5.70
2025-12-31T23:56:42Z,577609552,2025-12-31T18:56:42Z,MOD-00684,50.1,0.1,12.582,1.044,0.287,0.041,0.042,...,34.814,23.749,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,2.59
2025-12-31T23:55:42Z,577609550,2025-12-31T18:55:42Z,MOD-00684,49.8,0.1,11.556,0.975,0.286,0.045,0.041,...,33.879,25.193,14325.0,14326.0,14327.0,14471.0,14496.0,14546.0,14521.0,1.91


In [26]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:42Z,2025-12-31T18:59:42Z,828.982,2.560,33.422,25.941,11.578,0.908,0.229,0.048,0.052,0.018
2025-12-31T23:58:42Z,2025-12-31T18:58:42Z,875.709,2.768,33.418,25.930,11.323,0.940,0.248,0.054,0.089,0.018
2025-12-31T23:57:42Z,2025-12-31T18:57:42Z,1005.216,2.897,34.116,25.193,12.160,1.016,0.181,0.042,0.057,0.037
2025-12-31T23:56:42Z,2025-12-31T18:56:42Z,909.765,2.896,34.814,23.749,12.582,1.044,0.287,0.041,0.042,0.028
2025-12-31T23:55:42Z,2025-12-31T18:55:42Z,817.203,2.897,33.879,25.193,11.556,0.975,0.286,0.045,0.041,0.009


In [27]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:42Z,828.982,2.560,33.422,25.941,11.578,0.908,0.229,0.048,0.052,0.018
1,2025-12-31T18:58:42Z,875.709,2.768,33.418,25.930,11.323,0.940,0.248,0.054,0.089,0.018
2,2025-12-31T18:57:42Z,1005.216,2.897,34.116,25.193,12.160,1.016,0.181,0.042,0.057,0.037
3,2025-12-31T18:56:42Z,909.765,2.896,34.814,23.749,12.582,1.044,0.287,0.041,0.042,0.028
4,2025-12-31T18:55:42Z,817.203,2.897,33.879,25.193,11.556,0.975,0.286,0.045,0.041,0.009


In [28]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:42,828.982,2.560,33.422,25.941,11.578,0.908,0.229,0.048,0.052,0.018
1,2025-12-31 18:58:42,875.709,2.768,33.418,25.930,11.323,0.940,0.248,0.054,0.089,0.018
2,2025-12-31 18:57:42,1005.216,2.897,34.116,25.193,12.160,1.016,0.181,0.042,0.057,0.037
3,2025-12-31 18:56:42,909.765,2.896,34.814,23.749,12.582,1.044,0.287,0.041,0.042,0.028
4,2025-12-31 18:55:42,817.203,2.897,33.879,25.193,11.556,0.975,0.286,0.045,0.041,0.009


In [29]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,861.86,35.06,29.63,1.83,28.85,3.35,1.52,0.52,0.72,0.65
1,2025-03-31 21:00:00,1034.56,39.78,30.42,2.80,36.55,4.99,1.76,0.53,0.74,0.71
2,2025-03-31 22:00:00,1084.97,35.96,37.72,2.89,31.16,5.65,1.75,0.44,0.54,0.48
3,2025-03-31 23:00:00,804.92,28.57,45.88,2.90,23.08,6.01,1.58,0.32,0.36,0.26
4,2025-04-01 00:00:00,715.66,26.88,51.29,2.83,20.00,3.69,0.92,0.20,0.24,0.16
...,...,...,...,...,...,...,...,...,...,...,...
6571,2025-12-31 14:00:00,716.60,31.61,32.42,1.95,11.85,0.81,0.22,0.04,0.04,0.02
6572,2025-12-31 15:00:00,728.30,32.36,31.70,1.84,10.74,0.80,0.22,0.03,0.04,0.02
6573,2025-12-31 16:00:00,729.69,32.82,29.93,2.35,11.77,0.95,0.27,0.04,0.04,0.02
6574,2025-12-31 17:00:00,784.93,34.06,25.45,2.43,10.93,1.15,0.30,0.05,0.05,0.03


In [30]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,861.86,35.06,29.63,1.83,28.85,3.35,1.52,0.52,0.72,0.65
2025-03-31 21:00:00,1034.56,39.78,30.42,2.80,36.55,4.99,1.76,0.53,0.74,0.71
2025-03-31 22:00:00,1084.97,35.96,37.72,2.89,31.16,5.65,1.75,0.44,0.54,0.48
2025-03-31 23:00:00,804.92,28.57,45.88,2.90,23.08,6.01,1.58,0.32,0.36,0.26
2025-04-01 00:00:00,715.66,26.88,51.29,2.83,20.00,3.69,0.92,0.20,0.24,0.16


In [31]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [32]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [33]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.557009,0.678012,0.336246,0.029873,0.286523,0.097214,0.086758,0.116592,0.212389,0.317073
2025-03-31 21:00:00,0.668623,0.769290,0.345211,0.045707,0.362995,0.144806,0.100457,0.118834,0.218289,0.346341
2025-03-31 22:00:00,0.701202,0.695417,0.428053,0.047176,0.309465,0.163958,0.099886,0.098655,0.159292,0.234146
2025-03-31 23:00:00,0.520209,0.552504,0.520654,0.047339,0.229218,0.174405,0.090183,0.071749,0.106195,0.126829
2025-04-01 00:00:00,0.462522,0.519822,0.582047,0.046197,0.198629,0.107081,0.052511,0.044843,0.070796,0.078049


In [34]:
data.to_csv('MOD-00684_timeseries_hourly_scaled.csv')

In [35]:
start = data.index.min()

end = data.index.min()

print(start, end)

2025-03-31 20:00:00 2025-03-31 20:00:00


## Implementing NMF

In [36]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [37]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.563257,0.675148,0.342431,0.049116,0.233234,0.213029,0.147295,0.133266,0.176898,0.216904
2025-03-31 21:00:00,0.658030,0.774300,0.359034,0.056631,0.319589,0.254917,0.166344,0.145916,0.190965,0.233558
2025-03-31 22:00:00,0.663403,0.712187,0.448220,0.056230,0.307760,0.209791,0.130357,0.112227,0.148787,0.182750
2025-03-31 23:00:00,0.562289,0.533831,0.500865,0.047741,0.218747,0.143281,0.091364,0.080879,0.111576,0.137999
2025-04-01 00:00:00,0.536813,0.486920,0.549004,0.045068,0.167776,0.087836,0.053717,0.047939,0.070897,0.089364
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.478309,0.604288,0.360855,0.038996,0.108666,0.028065,0.008506,0.004974,0.011849,0.018440
2025-12-31 15:00:00,0.479778,0.621427,0.355150,0.039151,0.101446,0.025677,0.007782,0.004721,0.011669,0.018384
2025-12-31 16:00:00,0.480724,0.630616,0.335381,0.039122,0.112069,0.030316,0.009188,0.005059,0.011265,0.017727


In [38]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.064255,0.044576,0.038398,0.117164
1,0.073644,0.063184,0.034813,0.126162
2,0.069111,0.058493,0.065308,0.094566
3,0.052296,0.037631,0.095607,0.067892
4,0.049659,0.025345,0.112725,0.038373
...,...,...,...,...
6534,0.065427,0.016694,0.047322,0.000000
6535,0.067374,0.015274,0.044048,0.000000
6536,0.068249,0.018033,0.037678,0.000000
6537,0.072174,0.020498,0.023701,0.001691


In [39]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [40]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,5.442496,9.124069,2.975275,0.447968,0.234237,0.000000,0.000000,0.020630,0.100574,0.188446
Feature 2,2.524342,0.438697,0.000000,0.176395,4.590560,1.681136,0.509509,0.161438,0.043497,0.000000
Feature 3,1.692223,0.000000,3.511917,0.142459,0.353019,0.000000,0.000000,0.019641,0.095996,0.129129
Feature 4,0.307657,0.591685,0.140008,0.059737,0.000000,1.178609,1.063321,1.058261,1.406669,1.705620


In [41]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.064255,0.044576,0.038398,0.117164
2025-03-31 21:00:00,0.073644,0.063184,0.034813,0.126162
2025-03-31 22:00:00,0.069111,0.058493,0.065308,0.094566
2025-03-31 23:00:00,0.052296,0.037631,0.095607,0.067892
2025-04-01 00:00:00,0.049659,0.025345,0.112725,0.038373
...,...,...,...,...
2025-12-31 14:00:00,0.065427,0.016694,0.047322,0.000000
2025-12-31 15:00:00,0.067374,0.015274,0.044048,0.000000
2025-12-31 16:00:00,0.068249,0.018033,0.037678,0.000000


In [42]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,5.442496,9.124069,2.975275,0.447968,0.234237,0.000000,0.000000,0.020630,0.100574,0.188446
Factor 2,2.524342,0.438697,0.000000,0.176395,4.590560,1.681136,0.509509,0.161438,0.043497,0.000000
Factor 3,1.692223,0.000000,3.511917,0.142459,0.353019,0.000000,0.000000,0.019641,0.095996,0.129129
Factor 4,0.307657,0.591685,0.140008,0.059737,0.000000,1.178609,1.063321,1.058261,1.406669,1.705620


In [43]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.585795,0.134821,0.257393,0.009031,0.012961
no,0.590273,0.115334,0.265269,0.021467,0.007657
no2,0.965615,0.023038,0.000000,0.017077,-0.005731
o3,0.375565,0.000000,0.626460,0.004820,-0.006844
bin0,0.079201,0.770202,0.168680,0.000000,-0.018084
bin1,0.000000,0.821545,0.000000,0.316556,-0.138101
bin2,0.000000,0.492533,0.000000,0.564938,-0.057471
bin3,0.049480,0.192133,0.066571,0.692217,-0.000401
bin4,0.156223,0.033526,0.210720,0.595894,0.003637
bin5,0.224269,0.000000,0.217169,0.553581,0.004980


In [44]:
len(data)

6539

In [ ]:
results.to_csv('MOD-00684_factor_results.csv')
comp.to_csv('MOD-00684_factor_comp.csv')
res.to_csv('MOD-00684_factor_resid.csv')